# Creating PECT Dataset Prompts - Experiment 3 (LLM-based Context Selection)

This notebook implements the third set of experiments described in the dissertation:
instead of extracting every noun/verb/adjective via POS-tagging and querying the Google
Knowledge Graph (KG) for all of them (as in the Experiment 2 notebook), here an LLM
(Gemini) is used twice:

1. To **select the N most relevant terms** from the argument and topic (replacing the
   naive POS-tagging extraction).
2. To **prune the K candidates retrieved per term** from the Google KG Search API,
   keeping only the entries that are actually relevant to the argument-topic relation and
   discarding noise (e.g. a common word matching an unrelated popular entity).

Both `N_TERMS` and `K_DEPTH` are exposed as configuration variables below so the
experiment can be re-run with different combinations.

In [ ]:
# ===============================================================================
# CELL 1: INSTALLATION
# ===============================================================================
!pip install --quiet google-api-python-client google-generativeai pandas tqdm

In [ ]:
# ===============================================================================
# CELL 2: IMPORTS
# ===============================================================================
import os
import re
import json
import time
import pandas as pd
from google.colab import drive
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import google.generativeai as genai
from tqdm.auto import tqdm

In [ ]:
# ===============================================================================
# CELL 3: MOUNT GOOGLE DRIVE
# ===============================================================================
try:
    drive.mount('/content/drive')
except Exception:
    print("Drive already mounted or error mounting.")

In [ ]:
# ===============================================================================
# CELL 4: CONFIGURATION
# ===============================================================================

# --- API keys ---
API_KEY = ""         # Your Google Knowledge Graph API key here
GEMINI_API_KEY = ""  # Your Gemini API key here

# --- Experiment 3 parameters ---
N_TERMS = 5   # Number of terms the LLM selects from argument+topic to search in the KG
K_DEPTH = 3   # Number of candidate results retrieved per term from the Google KG API

# --- Paths ---
INPUT_CSV_PATH = '/content/drive/My Drive/ProjectFolder/aae_test.csv'  # Same input as Experiment 1/2 notebooks
OUTPUT_CSV_PATH = '/content/drive/My Drive/ProjectFolder/aae_test_prompts_experiment3.csv'

genai.configure(api_key=GEMINI_API_KEY)
selector_model = genai.GenerativeModel('gemini-1.5-flash')

if not API_KEY:
    print("WARNING: Google Knowledge Graph API key not configured.")
if not GEMINI_API_KEY:
    print("WARNING: Gemini API key not configured.")

## Term selection and knowledge graph pruning functions

In [ ]:
# ===============================================================================
# CELL 5: TERM SELECTION VIA LLM
# ===============================================================================

def _extract_json_block(text):
    """Strips optional markdown code fences around a JSON payload."""
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    return text


def select_relevant_terms(argument, topic, n_terms, model):
    """Asks the LLM to choose up to n_terms words/short phrases from the argument and
    topic that would benefit most from being looked up in a knowledge graph to help
    decide whether the argument attacks or supports the topic."""
    prompt = (
        f"You will select the {n_terms} most relevant single words or short phrases "
        f"(entities, concepts, or terms) from the ARGUMENT and TOPIC below that would "
        f"benefit most from being looked up in a knowledge graph to help decide whether "
        f"the argument attacks or supports the topic. "
        f"Return ONLY a JSON list of strings, no explanation.\n\n"
        f"Argument: \"{argument}\"\n"
        f"Topic: \"{topic}\"\n\n"
        f"JSON list of up to {n_terms} terms:"
    )
    for attempt in range(2):
        try:
            response = model.generate_content(prompt)
            text = _extract_json_block(response.text)
            terms = json.loads(text)
            if isinstance(terms, list):
                return [str(t).strip() for t in terms if str(t).strip()][:n_terms]
        except Exception:
            continue
    return []

In [ ]:
# ===============================================================================
# CELL 6: GOOGLE KNOWLEDGE GRAPH SEARCH (TOP-K CANDIDATES PER TERM)
# ===============================================================================

def search_kg_candidates(term, service, k_depth):
    """Searches the Google Knowledge Graph for up to k_depth candidate entities for a
    single term."""
    try:
        request = service.entities().search(query=term, languages='en', limit=k_depth)
        response = request.execute()
    except Exception:
        return []

    candidates = []
    for item in response.get('itemListElement', []):
        result = item.get('result', {})
        name = result.get('name', 'N/A')
        if 'detailedDescription' in result:
            description = result['detailedDescription'].get('articleBody', "No detailed description text.")
        elif 'description' in result:
            description = result.get('description')
        else:
            description = "No description available."
        candidates.append({'term': term, 'name': name, 'description': description})
    return candidates

In [ ]:
# ===============================================================================
# CELL 7: PRUNING OF RETRIEVED CANDIDATES VIA LLM
# ===============================================================================

def prune_candidates_with_llm(argument, topic, candidates, model):
    """Asks the LLM to judge which of the retrieved knowledge graph candidates are
    actually relevant to the argument-topic relation, discarding semantic noise
    (e.g. a common word matching an unrelated popular entity)."""
    if not candidates:
        return []

    numbered = "\n".join(
        f"{i}. Query='{c['term']}' -> Result='{c['name']}'. Definition: {c['description']}"
        for i, c in enumerate(candidates)
    )
    prompt = (
        f"Below is a list of candidate knowledge graph entries retrieved for terms "
        f"found in an ARGUMENT and a TOPIC. Some entries are relevant (they clarify a "
        f"concept actually used in the argument/topic reasoning) and some are noise "
        f"(unrelated popular entities that happen to share a name with the query term). "
        f"Return ONLY a JSON list of the indices (integers) of the RELEVANT entries.\n\n"
        f"Argument: \"{argument}\"\n"
        f"Topic: \"{topic}\"\n\n"
        f"Candidates:\n{numbered}\n\n"
        f"JSON list of relevant indices:"
    )
    for attempt in range(2):
        try:
            response = model.generate_content(prompt)
            text = _extract_json_block(response.text)
            indices = json.loads(text)
            if isinstance(indices, list):
                return [candidates[i] for i in indices if isinstance(i, int) and 0 <= i < len(candidates)]
        except Exception:
            continue
    # Fallback: if pruning fails twice, keep every candidate rather than losing context.
    return candidates

In [ ]:
# ===============================================================================
# CELL 8: FULL CONTEXT-BUILDING PIPELINE (SELECT -> SEARCH -> PRUNE)
# ===============================================================================

def build_llm_selected_context(argument, topic, kg_service, llm_model, n_terms, k_depth):
    """Runs the full Experiment 3 pipeline for a single argument-topic pair:
    1) select up to n_terms relevant terms via the LLM,
    2) search up to k_depth candidates per term in the Google KG,
    3) prune the candidates via the LLM, keeping only relevant entries.
    Returns the final context string and the list of selected terms."""
    terms = select_relevant_terms(argument, topic, n_terms, llm_model)

    all_candidates = []
    for term in terms:
        all_candidates.extend(search_kg_candidates(term, kg_service, k_depth))
        time.sleep(0.1)  # avoid overloading the Google KG API

    pruned = prune_candidates_with_llm(argument, topic, all_candidates, llm_model)

    if not pruned:
        return 'No relevant definitions found for the key concepts.', terms

    context_str = ' '.join(
        f"Query: '{c['term']}' -> Result: '{c['name']}'. Definition: {c['description']}"
        for c in pruned
    )
    return context_str, terms


def build_final_prompt(argument, topic, knowledge_context):
    question_prompt = f'The argument "{argument}" SUPPORT or ATTACK the belief "{topic}"?'
    return (
        f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
        f"which shows direct paths between concepts from the argument and the belief. "
        f"Based on this, decide if the argument attacks or supports the belief. "
        f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
        f"Knowledge Graph Context: {knowledge_context}\n\n"
        f"Question: {question_prompt}"
    )

## Single-case test (sanity check before batch processing)

In [ ]:
# ===============================================================================
# CELL 9: SINGLE CASE TEST
# ===============================================================================
print(f"--- Running test with the first row of the dataset '{os.path.basename(INPUT_CSV_PATH)}' ---")

if not API_KEY or not GEMINI_API_KEY:
    print("WARNING: One or both API keys are not configured.")
else:
    try:
        df_test = pd.read_csv(INPUT_CSV_PATH)
        example_row = df_test.iloc[0]

        argument_text = example_row['argument']
        topic_text = example_row['topic']

        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)

        start_time = time.time()
        knowledge_context, selected_terms = build_llm_selected_context(
            argument_text, topic_text, kg_service, selector_model, N_TERMS, K_DEPTH
        )
        end_time = time.time()

        final_prompt_example = build_final_prompt(argument_text, topic_text, knowledge_context)

        print(f"\nArgument: {argument_text}")
        print(f"Topic: {topic_text}")
        print(f"Selected terms (N={N_TERMS}): {selected_terms}")
        print(f"\n⏱️ Pipeline execution time: {end_time - start_time:.4f} seconds")
        print("\n--- Generated Prompt for the Example ---")
        print(final_prompt_example)

    except FileNotFoundError:
        print(f"ERROR: File not found at: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## Batch processing over the full dataset

In [ ]:
# ===============================================================================
# CELL 10: MAIN PROCESSING SCRIPT
# ===============================================================================

if not API_KEY or not GEMINI_API_KEY:
    print("WARNING: One or both API keys are not configured. The script cannot continue.")
else:
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
        print(f"File '{os.path.basename(INPUT_CSV_PATH)}' loaded successfully. Total of {len(df)} rows.")
    except FileNotFoundError:
        print(f"ERROR: File not found at the specified path: {INPUT_CSV_PATH}")
        df = None
    except Exception as e:
        print(f"An unexpected error occurred during loading: {e}")
        df = None

    if df is not None:
        results_data = []
        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)

        print(f"\nStarting Experiment 3 prompt generation for {len(df)} samples (N_TERMS={N_TERMS}, K_DEPTH={K_DEPTH}).")
        for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating Experiment 3 prompts"):
            argument_text = row['argument']
            topic_text = row['topic']

            knowledge_context, selected_terms = build_llm_selected_context(
                argument_text, topic_text, kg_service, selector_model, N_TERMS, K_DEPTH
            )
            final_prompt = build_final_prompt(argument_text, topic_text, knowledge_context)

            results_data.append({
                'ID': row['ID'],
                'SET': row['SET'],
                'topic': topic_text,
                'argument': argument_text,
                'selected_terms': json.dumps(selected_terms),
                'knowledge_context_llm': knowledge_context,
                'final_prompt_llm': final_prompt,
                'ground_truth_stance': row['classification'],
            })

        results_df = pd.DataFrame(results_data)
        results_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
        print(f"\n✅ Process completed! The file '{os.path.basename(OUTPUT_CSV_PATH)}' was saved successfully.")
        display(results_df.head())

In [ ]:
results_df.ground_truth_stance.value_counts()

In [ ]:
print(results_df.final_prompt_llm[0])